# Production Data Analysis
Анализ производственных данных: загрузка, очистка, KPI, визуализация и выводы.

**Файл данных:** `../data/production_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/production_data.csv', parse_dates=['date'])
print('shape:', df.shape)
display(df.head())
print(df.info())
print('\nПропуски:\n', df.isna().sum())
print('\nДубликатов:', df.duplicated().sum())

## 2. Очистка данных
- удаляем дубликаты
- пропуски в `defects` и `downtime_min` заполняем медианой
- приводим `date` к дате, сортируем

In [ ]:
before = len(df)
df_clean = df.drop_duplicates().copy()
for col in ['defects', 'downtime_min']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean = df_clean.sort_values('date').reset_index(drop=True)
# производные метрики
df_clean['fulfillment_pct'] = df_clean['produced_qty'] / df_clean['planned_qty'] * 100
df_clean['defect_rate_pct'] = df_clean['defects'] / df_clean['produced_qty'] * 100
print(f'Строк до: {before}, после: {len(df_clean)}')
print('Пропуски после очистки:\n', df_clean.isna().sum())
display(df_clean.describe().round(2))

## 3. KPI и группировки

In [ ]:
total_planned = df_clean['planned_qty'].sum()
total_produced = df_clean['produced_qty'].sum()
fulfillment = total_produced / total_planned * 100
defect_rate = df_clean['defects'].sum() / total_produced * 100
avg_downtime = df_clean['downtime_min'].mean()
energy_per_unit = df_clean['energy_kwh'].sum() / total_produced

print(f'Всего план: {total_planned}')
print(f'Всего факт: {total_produced}')
print(f'Выполнение плана: {fulfillment:.2f}%')
print(f'Доля брака: {defect_rate:.2f}%')
print(f'Средний простой: {avg_downtime:.1f} мин/партия')
print(f'Энергоёмкость: {energy_per_unit:.3f} кВтч/шт')

by_line = df_clean.groupby('line').agg(
    batches=('line', 'size'),
    fulfillment_pct=('fulfillment_pct', 'mean'),
    defect_rate_pct=('defect_rate_pct', 'mean'),
    downtime_min=('downtime_min', 'mean'),
).round(2)
print('\nПо линиям:')
display(by_line)

by_shift = df_clean.groupby('shift').agg(
    batches=('shift', 'size'),
    fulfillment_pct=('fulfillment_pct', 'mean'),
    defect_rate_pct=('defect_rate_pct', 'mean'),
    downtime_min=('downtime_min', 'mean'),
).round(2)
print('\nПо сменам:')
display(by_shift)

## 4. Визуализация

In [ ]:
# 1. Динамика выпуска по дням
daily = df_clean.groupby('date')[['planned_qty', 'produced_qty']].sum()
plt.figure()
plt.plot(daily.index, daily['planned_qty'], label='План')
plt.plot(daily.index, daily['produced_qty'], label='Факт')
plt.title('Динамика выпуска по дням')
plt.xlabel('Дата'); plt.ylabel('Штук')
plt.legend(); plt.tight_layout()
plt.savefig('../outputs/trend.png', dpi=120)
plt.show()

# 2. Средний брак по линиям
plt.figure()
df_clean.groupby('line')['defect_rate_pct'].mean().plot(kind='bar')
plt.title('Средняя доля брака по линиям, %')
plt.ylabel('%'); plt.tight_layout()
plt.savefig('../outputs/defects_by_line.png', dpi=120)
plt.show()

# 3. Средний простой по сменам
plt.figure()
df_clean.groupby('shift')['downtime_min'].mean().plot(kind='bar')
plt.title('Средний простой по сменам, мин')
plt.ylabel('мин'); plt.tight_layout()
plt.savefig('../outputs/downtime_by_shift.png', dpi=120)
plt.show()

# 4. Энергия vs выпуск
plt.figure()
plt.scatter(df_clean['produced_qty'], df_clean['energy_kwh'], alpha=0.5)
plt.title('Энергопотребление vs выпуск')
plt.xlabel('produced_qty'); plt.ylabel('energy_kwh')
plt.tight_layout()
plt.savefig('../outputs/energy_vs_output.png', dpi=120)
plt.show()

print('Графики сохранены в ../outputs/')

## 5. Выводы
- **Line-C** — самая проблемная линия: самая высокая доля брака и самое низкое выполнение плана. Кандидат №1 на аудит оборудования и процессов.
- **Ночная смена** даёт больше брака при сопоставимых простоях — проверить освещение, усталость, контроль качества ночью.
- **Простои** напрямую съедают выполнение плана — снижение среднего простоя даст самый быстрый прирост выпуска.
- Энергопотребление линейно растёт с выпуском — энергоёмкость стабильна, аномалий нет.